# Description

## Learn LangChain in 60 Minutes — API notebook

A mental model before we start:

- **LangChain** is the toolkit: prompts, models, tools, and composable building blocks ("runnables").
- **LangGraph** is the orchestrator: stateful graphs, routing, checkpointing/memory, and interrupts for human‑in‑the‑loop (HITL).
- **Deep Agents** is an optional, higher-level layer used later in this tutorial for “agent app” patterns
  (filesystem tools, todos, subagents, sandboxing, and HITL gates).

# Imports

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

import langchain
import langchain_core
import langgraph

import langchain_API_utils as ut

In [3]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s - %(message)s"
)
_LOG = logging.getLogger("learn_langchain.api")

ut.print_environment_info()
print(f"LLM_PROVIDER={os.getenv('LLM_PROVIDER', '(unset)')}")

python=3.12.3
platform=Linux-6.12.67-linuxkit-aarch64-with-glibc2.39
langchain=1.2.8
langchain_core=1.2.8
langgraph=unknown
LLM_PROVIDER=(unset)


## Model

These notebooks are provider-agnostic: you pick a provider in `.env`, and the helper function builds the right chat model.

Supported now:
- `openai`
- `anthropic`

In [9]:
import os

from dotenv import load_dotenv
load_dotenv("langchain.env")

if os.getenv("LANGSMITH_TRACING", "").strip().lower() in {"1", "true", "yes"}:
    _LOG.info("LangSmith tracing requested (LANGSMITH_TRACING=true).")

In [5]:
!cat langchain.env

# Copy this file to `.env` and fill in your credentials.
# Never commit `.env`.

# Chat provider selector used by the notebooks.
# Supported out-of-the-box: `openai`, `anthropic`
# Optional (requires extra package): `ollama` (`pip install langchain-ollama`)
LLM_PROVIDER=openai

# Provider-specific chat model.
LLM_MODEL=gpt-4.1-mini

# Make runs more reproducible.
LLM_TEMPERATURE=0

# ---- OpenAI ----
# Optional:
# OPENAI_BASE_URL=
# OPENAI_ORG_ID=

# ---- Anthropic ----
# ANTHROPIC_API_KEY=<set_anthropic_key_here>

# ---- Embeddings for the docs-RAG demo ----
# `auto` => OpenAI embeddings if OPENAI_API_KEY exists, else deterministic fake embeddings.
EMBEDDING_PROVIDER=auto
EMBEDDING_MODEL=text-embedding-3-small
FAKE_EMBED_DIM=256

# ---- LangSmith tracing (optional) ----
# LANGSMITH_TRACING=true
# LANGSMITH_API_KEY=<set_langsmith_key_here>
# LANGSMITH_PROJECT=langchain-langgraph-60min


In [10]:
llm = ut.get_chat_model()
llm

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0xffff706585c0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0xffff700f5d60>, root_client=<openai.OpenAI object at 0xffff70b4a900>, root_async_client=<openai.AsyncOpenAI object at 0xffff70223fb0>, model_name='gpt-4.1-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), request_timeout=60.0, stream_usage=True, max_retries=2)

## Local dataset (`data/T1_slice.csv`)

We’ll use a local CSV so the examples feel concrete.

In [11]:
from pathlib import Path
import shutil

import pandas as pd

# TODO(ai_gp): Move this to a function in *_utils.py
DATASET_PATH = Path("data/T1_slice.csv").resolve()
df = pd.read_csv(DATASET_PATH)
TIME_COL = "Date/Time"
if TIME_COL in df.columns:
    df[TIME_COL] = pd.to_datetime(
        df[TIME_COL], format="%d %m %Y %H:%M", errors="coerce"
    )

# Make the dataset visible to Deep Agents filesystem tools under `/workspace/...`.
WORKSPACE_DIR = Path("workspace").resolve()
WORKSPACE_DATA_DIR = WORKSPACE_DIR / "data"
WORKSPACE_DATA_DIR.mkdir(parents=True, exist_ok=True)
WORKSPACE_DATASET_PATH = WORKSPACE_DATA_DIR / "T1_slice.csv"
if not WORKSPACE_DATASET_PATH.exists():
    shutil.copyfile(str(DATASET_PATH), str(WORKSPACE_DATASET_PATH))


,Date/Time,LV ActivePower (kW),Wind Speed (m/s),Theoretical_Power_Curve (KWh),Wind Direction (°)
0,2018-01-01 00:00:00,380.047791,5.311336,416.328908,259.994904
1,2018-01-01 00:10:00,453.769196,5.672167,519.917511,268.641113
2,2018-01-01 00:20:00,306.376587,5.216037,390.900016,272.564789
3,2018-01-01 00:30:00,419.645905,5.659674,516.127569,271.258087
4,2018-01-01 00:40:00,380.650696,5.577941,491.702972,265.674286


In [ ]:
df.head(5)

In [12]:
# TODO(ai_gp): Add explanation.
DATASET_META = ut.build_dataset_meta(df)
DATASET_META

{'path': 'data/T1_slice.csv',
 'workspace_path': 'workspace/data/T1_slice.csv',
 'tool_path': '/workspace/data/T1_slice.csv',
 'n_rows': 100,
 'n_cols': 5,
 'columns': ['Date/Time',
  'LV ActivePower (kW)',
  'Wind Speed (m/s)',
  'Theoretical_Power_Curve (KWh)',
  'Wind Direction (°)'],
 'dtypes': {'Date/Time': 'datetime64[ns]',
  'LV ActivePower (kW)': 'float64',
  'Wind Speed (m/s)': 'float64',
  'Theoretical_Power_Curve (KWh)': 'float64',
  'Wind Direction (°)': 'float64'},
 'sample_rows': [{'Date/Time': Timestamp('2018-01-01 00:00:00'),
   'LV ActivePower (kW)': 380.047790527343,
   'Wind Speed (m/s)': 5.31133604049682,
   'Theoretical_Power_Curve (KWh)': 416.328907824861,
   'Wind Direction (°)': 259.994903564453},
  {'Date/Time': Timestamp('2018-01-01 00:10:00'),
   'LV ActivePower (kW)': 453.76919555664,
   'Wind Speed (m/s)': 5.67216682434082,
   'Theoretical_Power_Curve (KWh)': 519.917511061494,
   'Wind Direction (°)': 268.64111328125},
  {'Date/Time': Timestamp('2018-01-01 

## LCEL (LangChain Expression Language)

LCEL is a _pipe_ syntax for composing steps, like a Unix pipe (`a | b | c`):
- build a prompt
- call a model
- parse the result

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a concise tutor. Answer clearly."),
        ("human", "{question}"),
    ]
)
chain = prompt | llm | StrOutputParser()
chain.invoke({"question": "Explain LCEL in one sentence."})

2026-03-26 22:41:47,402 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'LCEL (Low-Cost Embedded Linux) is a lightweight, cost-effective Linux distribution designed for embedded systems with limited resources.'

## Runnables: invoke / batch / stream / RunnableParallel

A “runnable” is anything you can _call_.
LangChain standardizes that with a few common methods:

- `.invoke(input)` → one input, one output
- `.batch([inputs])` → many inputs at once (often more efficient)
- `.stream(input)` → yield partial outputs as they arrive
- `RunnableParallel(...)` → run independent chains side-by-side and combine the results

When you’re learning, it helps to treat runnables like functions — except they can be composed and configured.


In [14]:
from langchain_core.runnables import RunnableParallel

summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You write crisp summaries."),
        ("human", "Summarize in 3 bullets:\n\n{text}"),
    ]
)
risks_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You list caveats."),
        ("human", "List 3 risks/caveats:\n\n{text}"),
    ]
)

summary_chain = summary_prompt | llm | StrOutputParser()
risks_chain = risks_prompt | llm | StrOutputParser()

parallel = RunnableParallel(summary=summary_chain, risks=risks_chain)
parallel.invoke(
    {"text": "LangChain provides composable building blocks for LLM apps."},
    config={"max_concurrency": 2},
)

2026-03-26 22:53:10,247 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-26 22:53:10,342 INFO httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'summary': '- LangChain offers modular components to create applications using large language models (LLMs).  \n- It enables easy integration and combination of various functionalities for LLM-based solutions.  \n- The framework supports building complex, customizable LLM applications efficiently.',
 'risks': 'Here are three risks or caveats when using LangChain for building LLM applications:\n\n1. **Complexity and Learning Curve**: While LangChain offers powerful composable components, effectively integrating and orchestrating these building blocks can be complex, especially for developers new to LLMs or the framework itself. This may lead to longer development times or suboptimal implementations.\n\n2. **Dependency on External APIs and Models**: LangChain often relies on external language models and APIs (e.g., OpenAI, Hugging Face). This introduces risks related to API availability, rate limits, cost, and potential changes in API behavior that can affect your application’s stabilit

In [ ]:
questions = [
    {"question": "What is a tool in LangChain?"},
    {"question": "What is ToolNode in LangGraph?"},
    {"question": "What does InjectedState do?"},
]
chain.batch(questions, return_exceptions=True, config={"max_concurrency": 3})

In [ ]:
chunks = []
for chunk in chain.stream(
    {"question": "Give me a 2-bullet explanation of RunnableParallel."}
):
    chunks.append(chunk)
final = "".join(chunks)
final[:300] + ("..." if len(final) > 300 else "")

## Tools: `@tool` + ToolNode execution

A *tool* is a normal Python function with a schema.
The LLM can “ask” to call a tool (with arguments), and your code executes it.

Two ways you’ll see tools used:

1) **Directly** (call the function yourself)
2) **Inside a graph** via `ToolNode` (LangGraph executes any requested tool calls and feeds results back)

If you’re new: don’t worry about the message formats yet. Focus on the story:
"model asks for tool" → "we run tool" → "tool returns data" → "model continues".


In [ ]:
from langchain_core.messages import AIMessage
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode

tool_node = ToolNode([ut.mean, ut.zscore])

g = StateGraph(ut.ToolState)
g.add_node("tools", tool_node)
g.add_edge(START, "tools")
g.add_edge("tools", END)
graph = g.compile()

tool_calls = [
    {
        "name": "mean",
        "args": {"xs": [1, 2, 3, 4]},
        "id": "t1",
        "type": "tool_call",
    },
    {
        "name": "zscore",
        "args": {"xs": [9, 10, 10], "x": 10},
        "id": "t2",
        "type": "tool_call",
    },  # error (std=0)
]

out = graph.invoke({"messages": [AIMessage(content="", tool_calls=tool_calls)]})
[
    type(m).__name__ + ":" + (getattr(m, "content", "")[:80])
    for m in out["messages"]
]

## InjectedState: runtime-only args (system-owned)

Sometimes a tool needs access to *system-owned* context that the model shouldn’t be allowed to spoof.

`InjectedState` is the pattern for that:
- your tool signature includes an injected parameter
- LangGraph supplies it at runtime (not from the model’s JSON arguments)

Think of it like dependency injection:
- model controls: normal tool inputs
- system controls: injected inputs (state, stores, call IDs)


In [ ]:
import json

from langchain_core.messages import AIMessage
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode

tool_node = ToolNode([ut.dataset_brief])
g = StateGraph(ut.InjectedStateState)
g.add_node("tools", tool_node)
g.add_edge(START, "tools")
g.add_edge("tools", END)
graph = g.compile()

state_in: ut.InjectedStateState = {
    "dataset_meta": DATASET_META,
    "messages": [
        AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "dataset_brief",
                    "args": {
                        "question": "What columns exist and what is the sampling frequency?"
                    },
                    "id": "t1",
                    "type": "tool_call",
                }
            ],
        )
    ],
}
out = graph.invoke(state_in)
json.loads(out["messages"][-1].content)

## InjectedStore: injected persistent store handle

A store is a place to keep small bits of information across calls (like preferences, cached results, or “facts we’ve already extracted”).

`InjectedStore` lets a tool receive a store handle **without** the model being able to fabricate it.

In this tutorial we use `InMemoryStore` for simplicity, but the pattern generalizes to other persistence layers.


In [ ]:
from langchain_core.messages import AIMessage
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
tool_node = ToolNode([ut.save_pref, ut.load_pref])
g = StateGraph(ut.StoreState)
g.add_node("tools", tool_node)
g.add_edge(START, "tools")
g.add_edge("tools", END)
graph = g.compile(store=store)

out1 = graph.invoke(
    {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "save_pref",
                        "args": {
                            "user_id": "u1",
                            "key": "freq_hint",
                            "value": "1min",
                        },
                        "id": "t1",
                        "type": "tool_call",
                    }
                ],
            )
        ]
    }
)
out2 = graph.invoke(
    {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "load_pref",
                        "args": {"user_id": "u1", "key": "freq_hint"},
                        "id": "t2",
                        "type": "tool_call",
                    }
                ],
            )
        ]
    }
)
out1["messages"][-1].content, out2["messages"][-1].content

## Agent APIs used in `langchain.example.ipynb`

An *agent* is a loop: the model looks at the conversation + available tools, chooses an action, and repeats until it’s done.

In this tutorial we use a helper, `create_agent(...)`, to build a tool-calling agent quickly.
Later, in the examples notebook, you’ll see the same ideas expressed as explicit LangGraph loops.

If you ever feel confused, this heuristic helps:
- **LangChain agent helpers** get you started fast.
- **LangGraph** is what you reach for when you want full control (state, routing, memory, HITL).


In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=llm,
    tools=[ut.utc_now],
    system_prompt="Use tools when a tool can answer the question more reliably than guessing.",
)
out = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Call utc_now and return the exact value.")
        ]
    }
)
[(type(m).__name__, getattr(m, "content", "")[:120]) for m in out["messages"]][
    -4:
]

## Tool-calling output contract (reproducible handoff)

A practical pattern from production-style graph tutorials: ask agents to return a **reproducible call snippet** after using tools.

Why this helps:
- humans can verify what happened
- downstream automation can replay behavior
- handoffs between teammates become less ambiguous


In [ ]:
contract_agent = create_agent(
    model=llm,
    tools=[ut.utc_now],
    system_prompt=(
        "When time is requested, call utc_now. "
        "In your final answer, include a fenced python block with the exact tool call used."
    ),
)
contract_out = contract_agent.invoke(
    {
        "messages": [
            HumanMessage(content="What is the current UTC time? Use your tool.")
        ]
    }
)
print(getattr(contract_out["messages"][-1], "content", ""))

## Advanced agent tool plumbing: `AgentState`, `ToolRuntime`, `InjectedToolCallId`

This section is here for when you’re ready to peek “under the hood”.

The high-level story:
- tool calls happen inside a conversation
- each tool call has an ID
- LangGraph/LangChain pass runtime helpers so tools can update state and emit the right `ToolMessage`

If it feels advanced on a first read, that’s normal — the goal is to make the concepts *available*, not to memorize them.


In [ ]:
import json

from langchain.agents import create_agent

CustomState, extract_facts = ut.make_custom_state_and_tool()

supervisor = create_agent(
    llm,
    tools=[extract_facts],
    system_prompt="First call extract_facts, then summarize the returned facts.",
    state_schema=CustomState,
)

state = supervisor.invoke(
    {
        "messages": [
            {"role": "user", "content": "Text: LangGraph supports interrupts."}
        ],
        "user_prefs": {"tone": "formal"},
        "facts": [],
    }
)
{
    "facts": state.get("facts"),
    "last": getattr(state["messages"][-1], "content", "")[:160],
}

## Human-in-the-loop building block: `interrupt(...)` + resume

Sometimes an agent should *pause* and ask a human before doing something risky:
- deleting a file
- sending an email
- running a trade
- making an irreversible change

LangGraph’s low-level building block for this is `interrupt(value)`:

- The first time a node calls `interrupt(...)`, execution **stops** and the graph returns an `__interrupt__` payload.
- To continue, you call the graph again with `Command(resume=...)`.
- When the graph resumes, the node is **re-executed**, and `interrupt(...)` returns the human’s choice.

In the next cell we create a tiny file in `tmp_runs/hitl/` and only delete it if the human approves.


In [ ]:
from pathlib import Path

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

builder = StateGraph(ut.HITLState)
builder.add_node("propose", ut.propose_delete)
builder.add_node("delete", ut.do_delete)
builder.add_edge(START, "propose")
builder.add_edge("propose", "delete")
builder.add_edge("delete", END)
graph = builder.compile(checkpointer=MemorySaver())

tmp_dir = Path("tmp_runs/hitl").resolve()
tmp_dir.mkdir(parents=True, exist_ok=True)
victim = tmp_dir / "victim.txt"
victim.write_text("delete me", encoding="utf-8")

thread_id = "HITL_API_DEMO"
out1 = graph.invoke(
    {"target_path": str(victim), "decision": ""},
    config={"configurable": {"thread_id": thread_id}},
)
pending = (
    out1.get("__interrupt__", [])[0].value if "__interrupt__" in out1 else None
)
out2 = graph.invoke(
    Command(resume="approve"), config={"configurable": {"thread_id": thread_id}}
)
{"pending": pending, "victim_exists_after": victim.exists()}